# Gig Worker Welfare Policy Analysis in India

In [1]:
#IMPORTS
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

In [7]:
## Defining File Paths
RAW_PATH     = "data/raw"
CLEANED_PATH = "data/cleaned"

os.makedirs(CLEANED_PATH, exist_ok=True)

In [ ]:
#standardizing
STATE_NAME_MAP = {
    'Andaman ar'                           : 'Andaman and Nicobar Islands',
    'Andaman And Nicobar Islands'          : 'Andaman and Nicobar Islands',
    'Andhra Prac'                          : 'Andhra Pradesh',
    'Arunachal P'                          : 'Arunachal Pradesh',
    'Chhattisgar'                          : 'Chhattisgarh',
    'Dadra & Nagar Haveli and Daman & Diu': 'Dadra and Nagar Haveli and Daman and Diu',
    'Himachal Pr'                          : 'Himachal Pradesh',
    'Jammu and'                            : 'Jammu and Kashmir',
    'Jammu & Kashmir'                      : 'Jammu and Kashmir',
    'Lakshadwee'                           : 'Lakshadweep',
    'Madhya Pra'                           : 'Madhya Pradesh',
    'Maharashtr'                           : 'Maharashtra',
}


def standardise_states(df, col='state_name'):
    df[col] = df[col].astype(str).str.strip() #strips
    df[col] = df[col].replace(STATE_NAME_MAP) #replacing
    return df
 
def remove_total_row(df, col='state_name'):
    return df[~df[col].str.lower().str.contains('total|india|all states', na=False)] #Removing summary rows

## Dataset 1: Worker Gender Distribution

In [13]:
df_gender = pd.read_csv(os.path.join(RAW_PATH, "worker_gender_27.csv")) 
df_gender.head()

,Sl. No.,State/UT,Male,Female,Others
0,1,Andaman and Nicobar Islands,21,11.0,NaN
1,2,Andhra Pradesh,2729,2396.0,NaN
2,3,Arunachal Pradesh,43,28.0,NaN
3,4,Assam,5387,5422.0,1.0
4,5,Bihar,2205,2338.0,NaN


In [15]:
df_gender.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Sl. No.   37 non-null     object 
 1   State/UT  37 non-null     object 
 2   Male      37 non-null     int64  
 3   Female    36 non-null     float64
 4   Others    5 non-null      float64
dtypes: float64(2), int64(1), object(2)
memory usage: 1.6+ KB


In [16]:
df_gender.isnull().sum()

Sl. No.      0
State/UT     0
Male         0
Female       1
Others      32
dtype: int64

In [19]:
df_gender.duplicated().sum()

np.int64(0)

In [20]:
df_gender = df_gender.drop(columns=[df_gender.columns[0]]) #Dropping
df_gender.columns = ['state_name', 'gig_male', 'gig_female', 'gig_others'] #Renaming
df_gender = standardise_states(df_gender) #standardizing
df_gender = remove_total_row(df_gender) #Removing summary 

#Converts gender to integers
df_gender['gig_others'] = pd.to_numeric(df_gender['gig_others'].replace('NA', 0), errors='coerce').fillna(0).astype(int)
df_gender['gig_male']   = pd.to_numeric(df_gender['gig_male'],   errors='coerce').fillna(0).astype(int)
df_gender['gig_female'] = pd.to_numeric(df_gender['gig_female'], errors='coerce').fillna(0).astype(int)

#Computing totals and percentages
df_gender['total_gig_workers'] = df_gender['gig_male'] + df_gender['gig_female'] + df_gender['gig_others']
df_gender['female_gig_pct']    = (df_gender['gig_female'] / df_gender['total_gig_workers'].replace(0, 1) * 100).round(2)
df_gender['snapshot_date']     = '2025-03-27'

In [21]:
df_gender.to_csv(os.path.join(CLEANED_PATH, 'clean_gig_workers_gender.csv'), index=False) #Saving
df_gender.head()

,state_name,gig_male,gig_female,gig_others,total_gig_workers,female_gig_pct,snapshot_date
0,Andaman and Nicobar Islands,21,11,0,32,34.38,2025-03-27
1,Andhra Pradesh,2729,2396,0,5125,46.75,2025-03-27
2,Arunachal Pradesh,43,28,0,71,39.44,2025-03-27
3,Assam,5387,5422,1,10810,50.16,2025-03-27
4,Bihar,2205,2338,0,4543,51.46,2025-03-27


## Dataset 2: Worker Registration — 27 Mar 2025

In [22]:
df_unreg27 = pd.read_csv(os.path.join(RAW_PATH, 'worker_reg_27.csv'))
df_unreg27.head()

,Sl. No.,State/UT,Total Registrations
0,1,Andaman and Nicobar Islands,33107
1,2,Andhra Pradesh,8358480
2,3,Arunachal Pradesh,205341
3,4,Assam,7638612
4,5,Bihar,29837386


In [25]:
df_unreg27.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   state_name                       36 non-null     object
 1   total_unorganised_registrations  36 non-null     int64 
 2   snapshot_date                    36 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.1+ KB


In [26]:
df_unreg27.duplicated().sum()

np.int64(0)

In [23]:
df_unreg27 = df_unreg27.drop(columns=[df_unreg27.columns[0]]) #Dropping
df_unreg27.columns = ['state_name', 'total_unorganised_registrations']  #Renaming
df_unreg27 = standardise_states(df_unreg27) #standardizing
df_unreg27 = remove_total_row(df_unreg27) #Removing summary 

#Converts gender to integers
df_unreg27['total_unorganised_registrations'] = pd.to_numeric(df_unreg27['total_unorganised_registrations'], errors='coerce').fillna(0).astype(int)
df_unreg27['snapshot_date'] = '2025-03-27' 

In [24]:
df_unreg27.to_csv(os.path.join(CLEANED_PATH, 'clean_total_unorganised_27mar.csv'), index=False)
df_unreg27.head()

,state_name,total_unorganised_registrations,snapshot_date
0,Andaman and Nicobar Islands,33107,2025-03-27
1,Andhra Pradesh,8358480,2025-03-27
2,Arunachal Pradesh,205341,2025-03-27
3,Assam,7638612,2025-03-27
4,Bihar,29837386,2025-03-27


## Dataset 3: Worker Registration — 23 Mar 2025

In [28]:
df_unreg23 = pd.read_csv(os.path.join(RAW_PATH, 'worker_reg_23.csv'))
df_unreg23.head()

,Sl. No.,State/UT,Total Registrations
0,1,Andaman and Nicobar Islands,33085
1,2,Andhra Pradesh,8344403
2,3,Arunachal Pradesh,204987
3,4,Assam,7635682
4,5,Bihar,29826750


In [29]:
df_unreg23.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Sl. No.              37 non-null     object
 1   State/UT             37 non-null     object
 2   Total Registrations  37 non-null     int64 
dtypes: int64(1), object(2)
memory usage: 1020.0+ bytes


In [31]:
df_unreg23.duplicated().sum()

np.int64(0)

In [32]:
df_unreg23 = df_unreg23.drop(columns=[df_unreg23.columns[0]]) #Dropping
df_unreg23.columns = ['state_name', 'total_unorganised_registrations']  #Renaming
df_unreg23 = standardise_states(df_unreg23) #standardizing
df_unreg23 = remove_total_row(df_unreg23) #Removing summary


#Converts gender to integers
df_unreg23['total_unorganised_registrations'] = pd.to_numeric(df_unreg23['total_unorganised_registrations'], errors='coerce').fillna(0).astype(int)
df_unreg23['snapshot_date'] = '2025-03-23'

In [33]:
df_unreg23.to_csv(os.path.join(CLEANED_PATH, 'clean_total_unorganised_23mar.csv'), index=False)
df_unreg23.head()

,state_name,total_unorganised_registrations,snapshot_date
0,Andaman and Nicobar Islands,33085,2025-03-23
1,Andhra Pradesh,8344403,2025-03-23
2,Arunachal Pradesh,204987,2025-03-23
3,Assam,7635682,2025-03-23
4,Bihar,29826750,2025-03-23


## Dataset 4 : PLFS 

In [34]:
df_plfs = pd.read_csv(os.path.join(RAW_PATH, 'PLFS.csv'))
df_plfs.head()

,Sl. No.,State/UT,2019-20,2020-21,2021-22,2022-23,2023-24
0,1,Andhra Pradesh,4.7,4.1,4.2,4.1,4.1
1,2,Arunachal Pradesh,6.7,5.7,7.7,4.8,6.1
2,3,Assam,7.9,4.1,3.8,1.7,3.9
3,4,Bihar,5.3,4.7,6.0,3.9,3.0
4,5,Chhattisgarh,3.3,2.5,2.5,2.5,2.5


In [35]:
df_plfs = df_plfs.drop(columns=[df_plfs.columns[0]]) #Dropping
df_plfs.columns = ['state_name', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24'] #Renaming
df_plfs = standardise_states(df_plfs) #standardizing
df_plfs = remove_total_row(df_plfs) #Removing summary

#Reshaping
df_plfs_long = df_plfs.melt(
    id_vars    = ['state_name'],
    value_vars = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24'],
    var_name   = 'year',
    value_name = 'unemployment_rate_pct'
)
 
df_plfs_long['unemployment_rate_pct'] = pd.to_numeric(df_plfs_long['unemployment_rate_pct'], errors='coerce') ##Converts gender to integers

#Adding metadata columns
df_plfs_long['metric']      = 'unemployment_rate_usual_status_pct'
df_plfs_long['data_source'] = 'PLFS Annual Report MoSPI'
df_plfs_long = df_plfs_long.sort_values(['state_name', 'year']).reset_index(drop=True) #Sorting

In [36]:
df_plfs_long.to_csv(os.path.join(CLEANED_PATH, 'clean_plfs_unemployment.csv'), index=False)
df_plfs_long.head(10)

,state_name,year,unemployment_rate_pct,metric,data_source
0,Andaman and Nicobar Islands,2019-20,12.6,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
1,Andaman and Nicobar Islands,2020-21,9.1,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
2,Andaman and Nicobar Islands,2021-22,7.8,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
3,Andaman and Nicobar Islands,2022-23,9.7,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
4,Andaman and Nicobar Islands,2023-24,11.8,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
5,Andhra Pradesh,2019-20,4.7,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
6,Andhra Pradesh,2020-21,4.1,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
7,Andhra Pradesh,2021-22,4.2,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
8,Andhra Pradesh,2022-23,4.1,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI
9,Andhra Pradesh,2023-24,4.1,unemployment_rate_usual_status_pct,PLFS Annual Report MoSPI


## Dataset 5 :State-wise Population

In [37]:
df_pop = pd.read_csv(os.path.join(RAW_PATH, 'state_population_2025.csv'))

In [38]:
df_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   state_name      37 non-null     object 
 1   total_pop_2025  37 non-null     int64  
 2   pop_share_pct   37 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1020.0+ bytes


In [39]:
df_pop.duplicated().sum()

np.int64(0)

In [40]:
df_pop = standardise_states(df_pop) #standardizing
df_pop = remove_total_row(df_pop) ##Removing summary
 
df_pop.to_csv(os.path.join(CLEANED_PATH, 'clean_state_population.csv'), index=False)
df_pop.head()

,state_name,total_pop_2025,pop_share_pct
0,Uttar Pradesh,241265000,17.0206
1,Bihar,131041000,9.2446
2,Maharashtra,128659000,9.0765
3,West Bengal,100202000,7.0690
4,Madhya Pradesh,88985000,6.2776


## Dataset 6 :Policy Timeline

In [41]:
df_timeline = pd.read_csv(os.path.join(RAW_PATH, 'policy_timeline.csv'))
df_timeline.head()

,event_id,state_or_centre,policy_name,event_type,event_date,enforcement_status,notes
0,1,Central,Code on Wages 2019,Bill Passed,2019-08-08,Not Enforced,First central law mentioning gig workers
1,2,Central,Code on Social Security 2020,Bill Passed,2020-09-22,Not Enforced,Recognised gig workers as separate category fo...
2,3,Central,Code on Social Security 2020,Gazette Notified,2020-09-28,Not Enforced,Notified in gazette but never brought into for...
3,4,Central,e-Shram Portal Launch,Portal Launch,2021-08-26,Active,National registration portal for unorganised w...
4,5,Central,NITI Aayog Gig Economy Report,Policy Report,2022-06-27,Advisory Only,Estimated 7.7M gig workers in 2020-21 projecte...


In [43]:
df_timeline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   event_id            25 non-null     int64 
 1   state_or_centre     25 non-null     object
 2   policy_name         25 non-null     object
 3   event_type          25 non-null     object
 4   event_date          25 non-null     object
 5   enforcement_status  25 non-null     object
 6   notes               25 non-null     object
dtypes: int64(1), object(6)
memory usage: 1.5+ KB


In [47]:
df_timeline.duplicated().sum()

np.int64(0)

In [44]:
# changing dtype
df_timeline['event_date'] = pd.to_datetime(df_timeline['event_date'], errors='coerce')

In [48]:
passed  = df_timeline[df_timeline['event_type'] == 'Bill Passed'][['state_or_centre', 'event_date']].rename(columns={'event_date': 'passed_date'}) #Filtering
gazette = df_timeline[df_timeline['event_type'] == 'Gazette Notified'][['state_or_centre', 'event_date']].rename(columns={'event_date': 'gazette_date'})
 
df_lag = passed.merge(gazette, on='state_or_centre', how='inner') #Merging
df_lag['policy_lag_days'] = (df_lag['gazette_date'] - df_lag['passed_date']).dt.days

In [49]:
df_timeline.to_csv(os.path.join(CLEANED_PATH, 'clean_policy_timeline.csv'), index=False)
df_lag.to_csv(os.path.join(CLEANED_PATH, 'clean_policy_lag.csv'), index=False)
df_lag

,state_or_centre,passed_date,gazette_date,policy_lag_days
0,Central,2019-08-08,2020-09-28,417
1,Central,2020-09-22,2020-09-28,6
2,Rajasthan,2023-07-24,2023-08-01,8
3,Bihar,2025-05-01,2025-07-24,84
4,Jharkhand,2025-08-01,2026-01-06,158


## Dataset 7 :State Coverage

In [58]:
df_cov = pd.read_csv(os.path.join(RAW_PATH, 'state_coverage.csv'))
df_cov.head()

,state_name,has_gig_law,law_passed_date,gazette_date,enforcement_status,protection_status,key_features
0,Andhra Pradesh,No,NaN,NaN,No Law,No_Law,NaN
1,Arunachal Pradesh,No,NaN,NaN,No Law,No_Law,NaN
2,Assam,No,NaN,NaN,No Law,No_Law,NaN
3,Bihar,Yes,2025-05-01,2025-07-24,Enforced,Enforced,"Unique IDs, maternity leave, accident compensa..."
4,Chhattisgarh,No,NaN,NaN,No Law,No_Law,NaN


In [51]:
df_cov.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   state_name          36 non-null     object
 1   has_gig_law         36 non-null     object
 2   law_passed_date     6 non-null      object
 3   gazette_date        4 non-null      object
 4   enforcement_status  36 non-null     object
 5   protection_status   36 non-null     object
 6   key_features        10 non-null     object
dtypes: object(7)
memory usage: 2.1+ KB


In [53]:
df_cov.isnull().sum()

state_name             0
has_gig_law            0
law_passed_date       30
gazette_date          32
enforcement_status     0
protection_status      0
key_features          26
dtype: int64

In [55]:
df_cov.duplicated().sum()

np.int64(0)

In [59]:
# changing dtype
df_cov['has_passed_date']  = df_cov['law_passed_date'].notna().astype(int)
df_cov['has_gazette_date'] = df_cov['gazette_date'].notna().astype(int)
df_cov['has_features']     = df_cov['key_features'].notna().astype(int)

In [56]:
df_cov = standardise_states(df_cov) #standardizing
 
def count_features(text):
    if pd.isna(text) or str(text).strip() in ['None', 'nan', '']:
        return 0
    return len(str(text).split(','))
 
df_cov['welfare_features_count'] = df_cov['key_features'].apply(count_features)
df_cov['has_law_binary']         = (df_cov['has_gig_law'] == 'Yes').astype(int)

In [60]:
df_cov.to_csv(os.path.join(CLEANED_PATH, 'clean_state_coverage.csv'), index=False)
df_cov.head()

,state_name,has_gig_law,law_passed_date,gazette_date,enforcement_status,protection_status,key_features,has_passed_date,has_gazette_date,has_features
0,Andhra Pradesh,No,NaN,NaN,No Law,No_Law,NaN,0,0,0
1,Arunachal Pradesh,No,NaN,NaN,No Law,No_Law,NaN,0,0,0
2,Assam,No,NaN,NaN,No Law,No_Law,NaN,0,0,0
3,Bihar,Yes,2025-05-01,2025-07-24,Enforced,Enforced,"Unique IDs, maternity leave, accident compensa...",1,1,1
4,Chhattisgarh,No,NaN,NaN,No Law,No_Law,NaN,0,0,0


## Dataset 8 :State-wise NSDP

In [61]:
df_nsdp = pd.read_csv(os.path.join(RAW_PATH, 'state_nsdp_2024_25.csv'))
df_nsdp.head()

,state_name,nsdp_crore_2024_25,per_capita_nsdp_rs,economy_size_category,data_source,data_year,unit,data_confidence,nsdp_share_of_india_pct,per_capita_vs_india_avg_pct,nsdp_rank,per_capita_rank,per_capita_status
0,Maharashtra,395731,309340,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,13.03,167.2,1,13,Above National Avg
1,Tamil Nadu,279249,361619,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,9.20,195.5,2,10,Above National Avg
2,Karnataka,260394,380906,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,8.58,205.9,3,8,Above National Avg
3,Uttar Pradesh,260000,108572,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,8.56,58.7,4,34,Below National Avg
4,Gujarat,245000,330000,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Estimated from RBI trend,8.07,178.4,5,12,Above National Avg


In [62]:
df_nsdp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   state_name                   36 non-null     object 
 1   nsdp_crore_2024_25           36 non-null     int64  
 2   per_capita_nsdp_rs           36 non-null     int64  
 3   economy_size_category        36 non-null     object 
 4   data_source                  36 non-null     object 
 5   data_year                    36 non-null     object 
 6   unit                         36 non-null     object 
 7   data_confidence              36 non-null     object 
 8   nsdp_share_of_india_pct      36 non-null     float64
 9   per_capita_vs_india_avg_pct  36 non-null     float64
 10  nsdp_rank                    36 non-null     int64  
 11  per_capita_rank              36 non-null     int64  
 12  per_capita_status            36 non-null     object 
dtypes: float64(2), int64(4

In [63]:
df_nsdp.isnull().sum()

state_name                     0
nsdp_crore_2024_25             0
per_capita_nsdp_rs             0
economy_size_category          0
data_source                    0
data_year                      0
unit                           0
data_confidence                0
nsdp_share_of_india_pct        0
per_capita_vs_india_avg_pct    0
nsdp_rank                      0
per_capita_rank                0
per_capita_status              0
dtype: int64

In [65]:
df_nsdp = standardise_states(df_nsdp) #Standardizing

df_nsdp.to_csv(os.path.join(CLEANED_PATH, 'clean_state_nsdp.csv'), index=False)
df_nsdp.head()

,state_name,nsdp_crore_2024_25,per_capita_nsdp_rs,economy_size_category,data_source,data_year,unit,data_confidence,nsdp_share_of_india_pct,per_capita_vs_india_avg_pct,nsdp_rank,per_capita_rank,per_capita_status
0,Maharashtra,395731,309340,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,13.03,167.2,1,13,Above National Avg
1,Tamil Nadu,279249,361619,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,9.20,195.5,2,10,Above National Avg
2,Karnataka,260394,380906,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,8.58,205.9,3,8,Above National Avg
3,Uttar Pradesh,260000,108572,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Verified,8.56,58.7,4,34,Below National Avg
4,Gujarat,245000,330000,Very Large (>2L Cr),RBI Handbook of Statistics on Indian States 20...,2024-25,Rs. Crore (Current Prices),Estimated from RBI trend,8.07,178.4,5,12,Above National Avg


## Creating Master Dataset

In [70]:
print(df_cov.columns)

Index(['state_name', 'has_gig_law', 'law_passed_date', 'gazette_date',
       'enforcement_status', 'protection_status', 'key_features',
       'has_passed_date', 'has_gazette_date', 'has_features',
       'welfare_features_count', 'has_law_binary'],
      dtype='object')


In [68]:
# add missing columns to df_cov before building master table

def count_features(text):
    if pd.isna(text) or str(text).strip() in ['None', 'nan', '']:
        return 0
    return len(str(text).split(','))

df_cov['welfare_features_count'] = df_cov['key_features'].apply(count_features)
df_cov['has_law_binary']         = (df_cov['has_gig_law'] == 'Yes').astype(int)

df_cov.columns.tolist()

['state_name',
 'has_gig_law',
 'law_passed_date',
 'gazette_date',
 'enforcement_status',
 'protection_status',
 'key_features',
 'has_passed_date',
 'has_gazette_date',
 'has_features',
 'welfare_features_count',
 'has_law_binary']

In [69]:
# Select key policy-related columns from df_cov and create a base master DataFrame
master = df_cov[['state_name', 'has_gig_law', 'has_law_binary','protection_status', 'welfare_features_count', 'enforcement_status']].copy()

# Merge gender distribution of gig workers (male, female, others, totals, female %)
master = master.merge(df_gender[['state_name', 'gig_male', 'gig_female', 'gig_others', 'total_gig_workers', 'female_gig_pct']],
    on='state_name', how='left')

# Merge unorganised worker registrations snapshot as of 27 March
master = master.merge(df_unreg27[['state_name', 'total_unorganised_registrations']].rename(columns={'total_unorganised_registrations': 
    'total_unorganised_27mar'}),on='state_name', how='left')

# Merge unorganised worker registrations snapshot as of 23 March
master = master.merge(df_unreg23[['state_name', 'total_unorganised_registrations']].rename(columns={'total_unorganised_registrations':
    'total_unorganised_23mar'}),on='state_name', how='left')

# Merge cleaned population data (total population and share % for 2025)
master = master.merge(df_pop[['state_name', 'total_pop_2025', 'pop_share_pct']],
    on='state_name', how='left')

# Merge Net State Domestic Product (NSDP) data including per capita values and ranks
master = master.merge(df_nsdp[['state_name', 'nsdp_crore_2024_25', 'per_capita_nsdp_rs', 'nsdp_rank', 'per_capita_rank', 'per_capita_status']],
    on='state_name', how='left')
 
master.head()

,state_name,has_gig_law,has_law_binary,protection_status,welfare_features_count,enforcement_status,gig_male,gig_female,gig_others,total_gig_workers,female_gig_pct,total_unorganised_27mar,total_unorganised_23mar,total_pop_2025,pop_share_pct,nsdp_crore_2024_25,per_capita_nsdp_rs,nsdp_rank,per_capita_rank,per_capita_status
0,Andhra Pradesh,No,0,No_Law,0,No Law,2729,2396,0,5125,46.75,8358480,8344403,53586000,3.7803,142299,266240,8,19,Above National Avg
1,Arunachal Pradesh,No,0,No_Law,0,No Law,43,28,0,71,39.44,205341,204987,1594000,0.1125,3800,238400,31,20,Above National Avg
2,Assam,No,0,No_Law,0,No Law,5387,5422,1,10810,50.16,7638612,7635682,36493000,2.5745,50000,137000,17,32,Below National Avg
3,Bihar,Yes,1,Enforced,5,Enforced,2205,2338,0,4543,51.46,29837386,29826750,131041000,9.2446,89902,69321,14,36,Below National Avg
4,Chhattisgarh,No,0,No_Law,0,No Law,437,210,0,647,32.46,8557694,8555759,30982000,2.1857,46000,148500,18,31,Below National Avg


In [75]:
# Gig registration rate 
master['gig_registration_rate_pct'] = (master['total_gig_workers'] / master['total_unorganised_27mar'].replace(0, 1) * 100).round(4)

# Protection gap
master['protection_gap']            = master['total_unorganised_27mar'] - master['total_gig_workers']

#short-term registration growth
master['reg_growth_4day']           = master['total_unorganised_27mar'] - master['total_unorganised_23mar']

#Gig workers per lakh population
master['gig_per_lakh_pop']          = (master['total_gig_workers'] / (master['total_pop_2025'] / 100000)).round(2)

master[['state_name', 'total_gig_workers', 'protection_gap', 'gig_registration_rate_pct', 'gig_per_lakh_pop','reg_growth_4day']].head(10)

,state_name,total_gig_workers,protection_gap,gig_registration_rate_pct,gig_per_lakh_pop,reg_growth_4day
0,Andhra Pradesh,5125,8353355,0.0613,9.56,14077
1,Arunachal Pradesh,71,205270,0.0346,4.45,354
2,Assam,10810,7627802,0.1415,29.62,2930
3,Bihar,4543,29832843,0.0152,3.47,10636
4,Chhattisgarh,647,8557047,0.0076,2.09,1935
5,Delhi,3745,3531211,0.1059,16.81,3082
6,Goa,104,77855,0.1334,6.53,52
7,Gujarat,4414,11966870,0.0369,6.00,3937
8,Haryana,1633,5377759,0.0304,5.26,1434
9,Himachal Pradesh,262,1991080,0.0132,3.47,572


In [72]:
#Law score: 30 points if a state has a gig law 
master['idx_law_score'] = master['has_law_binary'] * 30

#features score: scale welfare features count to a maximum of 20 points
master['idx_features_score'] = (master['welfare_features_count'] / master['welfare_features_count'].max() * 20).round(2)

# registration score: scale gig registration rate to a maximum of 25 points
master['idx_registration_score'] = (master['gig_registration_rate_pct'] / max(master['gig_registration_rate_pct'].max(), 0.0001) * 25).round(2)

#income score: scale per capita NSDP to a maximum of 25 points
master['idx_income_score'] = (master['per_capita_nsdp_rs'] / master['per_capita_nsdp_rs'].max() * 25).round(2)
 
#composite Gig Protection Index (sum of all four dimensions)
master['gig_protection_index'] = (master['idx_law_score'] + master['idx_features_score'] + master['idx_registration_score'] + master['idx_income_score']).round(2)

#Rank states by Gig Protection Index 
master['protection_index_rank'] = master['gig_protection_index'].rank(ascending=False, method='min').astype(int)

#EXPORTING
master.to_csv(os.path.join(CLEANED_PATH, 'master_analytics_table.csv'), index=False)
 
master[['state_name', 'gig_protection_index', 'protection_index_rank', 'protection_status']].sort_values('protection_index_rank').reset_index(drop=True)

,state_name,gig_protection_index,protection_index_rank,protection_status
0,Telangana,69.08,1,Enforced
1,Karnataka,68.66,2,Enforced
2,Rajasthan,61.69,3,Enforced
3,Jharkhand,53.05,4,Enforced
4,Bihar,51.76,5,Enforced
5,Goa,48.57,6,No_Law
6,Delhi,39.48,7,Discussions
7,Sikkim,39.43,8,No_Law
8,Assam,29.75,9,No_Law
9,Chandigarh,29.29,10,No_Law


In [74]:
master.shape

(36, 30)

In [73]:
master.duplicated().sum()

np.int64(0)

## Data Quality Checks
- Removed summary rows and columns
- Standardized state names
- Checked duplicates
- Validated merge consistency
- Converted data types
- Created new columns